In [0]:
%run ../99_utils/utils_config

UTILS CONFIG LOADED SUCCESSFULLY
PROJECT_NAME: brazil_legislative_analytics
PROJECT_VERSION: v1.0.0
PROJECT_ENVIRONMENT: dev
CATALOG_NAME: brazil_legislative_analytics
RUN_ID: 2a34f1fc-c4b2-4b28-8d94-66b55f988fcc


In [0]:
%run ../99_utils/utils_quality

UTILS CONFIG LOADED SUCCESSFULLY
PROJECT_NAME: brazil_legislative_analytics
PROJECT_VERSION: v1.0.0
PROJECT_ENVIRONMENT: dev
CATALOG_NAME: brazil_legislative_analytics
RUN_ID: b4581acb-ac78-4a69-963e-1f4f5390c58b


utils_quality loaded successfully.


In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 01 Quality — Bronze Checks
# MAGIC
# MAGIC Executes data quality validations for Bronze tables in the Brazil Legislative Analytics Medallion project.
# MAGIC
# MAGIC ## Purpose
# MAGIC This notebook validates whether Bronze ingestion outputs are available and structurally consistent before Silver processing.
# MAGIC
# MAGIC ## Scope
# MAGIC Bronze quality checks focus on:
# MAGIC - table existence
# MAGIC - required traceability columns
# MAGIC - minimum record availability
# MAGIC - duplicated hash records
# MAGIC - source endpoint traceability
# MAGIC - controlled exception handling per entity
# MAGIC
# MAGIC ## Persistence
# MAGIC Validation results are persisted into:
# MAGIC
# MAGIC ```text
# MAGIC audit.aud_log_qualidade_dados
# MAGIC ```
# MAGIC
# MAGIC ## Execution Policy
# MAGIC During early development, Bronze tables may not exist yet.
# MAGIC In this case, validations are persisted as evidence, but the notebook does not block execution.
# MAGIC
# MAGIC Set `FAIL_ON_ERROR = True` when Bronze ingestion is active and quality checks must block the pipeline.
# MAGIC
# MAGIC ## Documentation Standard
# MAGIC - Python functions and variables are written in English.
# MAGIC - Table and field names follow Portuguese mnemonic standards.
# MAGIC - Comments and documentation are written in English.

# COMMAND ----------

# MAGIC %run ../99_utils/utils_config

# COMMAND ----------

# MAGIC %run ../99_utils/utils_quality

# COMMAND ----------

from datetime import datetime
from typing import Dict, Any

from pyspark.sql import DataFrame
from pyspark.sql import functions as F

# COMMAND ----------

print("=" * 90)
print("BRAZIL LEGISLATIVE ANALYTICS MEDALLION")
print("01 - QUALITY BRONZE CHECKS")
print("=" * 90)
print(f"Execution Timestamp: {datetime.now()}")
print(f"Catalog: {CATALOG_NAME}")
print(f"Layer: {SCHEMA_BRONZE}")
print("=" * 90)

# COMMAND ----------

# ============================================================
# QUALITY CONFIGURATION
# ============================================================

NOTEBOOK_NAME = "01_quality_bronze_checks"
LAYER_NAME = "bronze"

# During early development, Bronze tables may not exist yet.
# Set to True when Bronze ingestion is active and quality checks
# must block the pipeline.
FAIL_ON_ERROR = False

DATA_QUALITY_LOG_TABLE = (
    f"{CATALOG_NAME}."
    f"{SCHEMA_AUDIT}."
    f"{AUD_TB_LOG_QUALIDADE_DADOS}"
)

BRONZE_ENTITY_TABLES = BRONZE_TABLES

quality_results = []

# COMMAND ----------

# ============================================================
# QUALITY HELPERS
# ============================================================

def add_quality_result(
    rule_name: str,
    rule_description: str,
    validation_status: str,
    total_records: int,
    invalid_records: int,
    invalid_percentage: float,
    message: str,
    entity_name: str,
    target_table: str,
) -> None:
    """
    Adds a quality validation result to the in-memory result list.
    """

    quality_results.append({
        "nome_regra": rule_name,
        "descricao_regra": rule_description,
        "status_validacao": validation_status,
        "total_registros": int(total_records) if total_records is not None else 0,
        "registros_invalidos": int(invalid_records) if invalid_records is not None else 0,
        "percentual_invalidos": float(invalid_percentage) if invalid_percentage is not None else 0.0,
        "mensagem": message,
        "entity_name": entity_name,
        "target_table": target_table,
    })

# COMMAND ----------

def add_exception_result(
    entity_name: str,
    target_table: str,
    error: Exception,
) -> None:
    """
    Adds a controlled exception result to the quality result list.
    """

    add_quality_result(
        rule_name="bronze_quality_exception",
        rule_description="Captures unexpected errors during Bronze quality validation.",
        validation_status=QUALITY_FAILED,
        total_records=1,
        invalid_records=1,
        invalid_percentage=100.0,
        message=f"Unexpected error during Bronze quality validation: {str(error)}",
        entity_name=entity_name,
        target_table=target_table,
    )

# COMMAND ----------

def table_exists(
    full_table_name: str,
) -> bool:
    """
    Checks whether a fully qualified table exists.
    """

    try:
        return spark.catalog.tableExists(full_table_name)

    except Exception:
        return False

# COMMAND ----------

def get_table_dataframe(
    full_table_name: str,
) -> DataFrame:
    """
    Reads a table into a Spark DataFrame.
    """

    return spark.table(full_table_name)

# COMMAND ----------

def count_records(
    dataframe: DataFrame,
) -> int:
    """
    Counts records from a Spark DataFrame.
    """

    return dataframe.count()

# COMMAND ----------

def validate_table_exists(
    entity_name: str,
    full_table_name: str,
) -> bool:
    """
    Validates whether a Bronze table exists.
    """

    exists = table_exists(full_table_name)

    add_quality_result(
        rule_name="bronze_table_exists",
        rule_description="Validates whether the Bronze table exists.",
        validation_status=QUALITY_PASSED if exists else QUALITY_FAILED,
        total_records=1,
        invalid_records=0 if exists else 1,
        invalid_percentage=0.0 if exists else 100.0,
        message=(
            "Bronze table exists."
            if exists
            else "Bronze table does not exist."
        ),
        entity_name=entity_name,
        target_table=full_table_name,
    )

    return exists

# COMMAND ----------

def validate_minimum_records(
    dataframe: DataFrame,
    entity_name: str,
    full_table_name: str,
) -> int:
    """
    Validates whether a Bronze table contains at least one record.
    """

    total_records = count_records(dataframe)
    invalid_records = 0 if total_records > 0 else 1

    add_quality_result(
        rule_name="bronze_minimum_records",
        rule_description="Validates whether the Bronze table contains at least one record.",
        validation_status=QUALITY_PASSED if total_records > 0 else QUALITY_WARNING,
        total_records=total_records,
        invalid_records=invalid_records,
        invalid_percentage=0.0 if total_records > 0 else 100.0,
        message=f"Bronze table record count: {total_records}",
        entity_name=entity_name,
        target_table=full_table_name,
    )

    return total_records

# COMMAND ----------

def validate_traceability_columns(
    dataframe: DataFrame,
    entity_name: str,
    full_table_name: str,
) -> None:
    """
    Validates required Bronze traceability columns.
    """

    result = validate_required_columns(
        dataframe=dataframe,
        required_columns=BRONZE_REQUIRED_COLUMNS,
    )

    add_quality_result(
        rule_name="bronze_required_traceability_columns",
        rule_description="Validates required Bronze traceability columns.",
        validation_status=result["status_validacao"],
        total_records=result["total_registros"],
        invalid_records=result["registros_invalidos"],
        invalid_percentage=result["percentual_invalidos"],
        message=result["mensagem"],
        entity_name=entity_name,
        target_table=full_table_name,
    )

# COMMAND ----------

def validate_endpoint_traceability(
    dataframe: DataFrame,
    entity_name: str,
    full_table_name: str,
) -> None:
    """
    Validates whether source endpoint traceability is populated.
    """

    if "aud_tx_endpoint_origem" not in dataframe.columns:

        add_quality_result(
            rule_name="bronze_endpoint_traceability",
            rule_description="Validates whether source endpoint traceability is available.",
            validation_status=QUALITY_FAILED,
            total_records=1,
            invalid_records=1,
            invalid_percentage=100.0,
            message="Column aud_tx_endpoint_origem does not exist.",
            entity_name=entity_name,
            target_table=full_table_name,
        )

        return

    total_records = dataframe.count()

    invalid_records = (
        dataframe
        .filter(
            F.col("aud_tx_endpoint_origem").isNull()
            | (F.trim(F.col("aud_tx_endpoint_origem")) == "")
        )
        .count()
    )

    invalid_percentage = (
        0.0 if total_records == 0
        else round((invalid_records / total_records) * 100, 2)
    )

    validation_status = (
        QUALITY_PASSED
        if invalid_records == 0
        else QUALITY_WARNING
    )

    add_quality_result(
        rule_name="bronze_endpoint_traceability",
        rule_description="Validates whether source endpoint traceability is populated.",
        validation_status=validation_status,
        total_records=total_records,
        invalid_records=invalid_records,
        invalid_percentage=invalid_percentage,
        message=f"Records without endpoint traceability: {invalid_records}",
        entity_name=entity_name,
        target_table=full_table_name,
    )

# COMMAND ----------

def validate_hash_duplicates(
    dataframe: DataFrame,
    entity_name: str,
    full_table_name: str,
) -> None:
    """
    Validates duplicated records based on the Bronze record hash.
    """

    if "aud_tx_hash_registro" not in dataframe.columns:

        add_quality_result(
            rule_name="bronze_hash_duplicate_check",
            rule_description="Validates duplicated records based on the Bronze record hash.",
            validation_status=QUALITY_FAILED,
            total_records=1,
            invalid_records=1,
            invalid_percentage=100.0,
            message="Column aud_tx_hash_registro does not exist.",
            entity_name=entity_name,
            target_table=full_table_name,
        )

        return

    result = validate_duplicates(
        dataframe=dataframe,
        key_columns=["aud_tx_hash_registro"],
    )

    add_quality_result(
        rule_name="bronze_hash_duplicate_check",
        rule_description="Validates duplicated records based on the Bronze record hash.",
        validation_status=result["status_validacao"],
        total_records=result["total_registros"],
        invalid_records=result["registros_invalidos"],
        invalid_percentage=result["percentual_invalidos"],
        message=result["mensagem"],
        entity_name=entity_name,
        target_table=full_table_name,
    )

# COMMAND ----------

def run_entity_checks(
    entity_name: str,
    table_name: str,
) -> None:
    """
    Executes all Bronze quality checks for a single entity.
    """

    full_table_name = get_bronze_table(table_name)

    print("=" * 90)
    print(f"Running Bronze quality checks for: {full_table_name}")
    print("=" * 90)

    try:

        if not validate_table_exists(
            entity_name=entity_name,
            full_table_name=full_table_name,
        ):
            return

        dataframe = get_table_dataframe(full_table_name)

        validate_minimum_records(
            dataframe=dataframe,
            entity_name=entity_name,
            full_table_name=full_table_name,
        )

        validate_traceability_columns(
            dataframe=dataframe,
            entity_name=entity_name,
            full_table_name=full_table_name,
        )

        validate_endpoint_traceability(
            dataframe=dataframe,
            entity_name=entity_name,
            full_table_name=full_table_name,
        )

        validate_hash_duplicates(
            dataframe=dataframe,
            entity_name=entity_name,
            full_table_name=full_table_name,
        )

    except Exception as error:

        add_exception_result(
            entity_name=entity_name,
            target_table=full_table_name,
            error=error,
        )

# COMMAND ----------

def build_bronze_quality_log() -> DataFrame:
    """
    Builds the final Bronze quality log DataFrame.
    """

    if not quality_results:

        add_quality_result(
            rule_name="bronze_quality_no_results",
            rule_description="Validates whether Bronze quality checks produced results.",
            validation_status=QUALITY_WARNING,
            total_records=0,
            invalid_records=0,
            invalid_percentage=0.0,
            message="No Bronze quality results were generated.",
            entity_name="bronze",
            target_table=DATA_QUALITY_LOG_TABLE,
        )

    quality_base_df = spark.createDataFrame(
        quality_results
    )

    return (
        quality_base_df
        .withColumn("qlt_id_log", F.expr("uuid()"))
        .withColumn("aud_id_execucao", F.lit(RUN_ID))
        .withColumn("aud_tx_nome_projeto", F.lit(PROJECT_NAME))
        .withColumn("aud_tx_versao_pipeline", F.lit(PROJECT_VERSION))
        .withColumn("aud_tx_ambiente", F.lit(PROJECT_ENVIRONMENT))
        .withColumn("aud_tx_nome_notebook", F.lit(NOTEBOOK_NAME))
        .withColumn("aud_tx_nome_camada", F.lit(LAYER_NAME))
        .withColumn("aud_tx_nome_entidade", F.col("entity_name"))
        .withColumn("aud_tx_tabela_destino", F.col("target_table"))
        .withColumn("qlt_tx_nome_regra", F.col("nome_regra"))
        .withColumn("qlt_tx_descricao_regra", F.col("descricao_regra"))
        .withColumn("qlt_tx_status_validacao", F.col("status_validacao"))
        .withColumn("qlt_qt_total_registros", F.col("total_registros"))
        .withColumn("qlt_qt_registros_invalidos", F.col("registros_invalidos"))
        .withColumn("qlt_pc_registros_invalidos", F.col("percentual_invalidos"))
        .withColumn("qlt_dh_validacao", F.current_timestamp())
        .withColumn("qlt_tx_mensagem", F.col("mensagem"))
        .select(
            "qlt_id_log",
            "aud_id_execucao",
            "aud_tx_nome_projeto",
            "aud_tx_versao_pipeline",
            "aud_tx_ambiente",
            "aud_tx_nome_notebook",
            "aud_tx_nome_camada",
            "aud_tx_nome_entidade",
            "aud_tx_tabela_destino",
            "qlt_tx_nome_regra",
            "qlt_tx_descricao_regra",
            "qlt_tx_status_validacao",
            "qlt_qt_total_registros",
            "qlt_qt_registros_invalidos",
            "qlt_pc_registros_invalidos",
            "qlt_dh_validacao",
            "qlt_tx_mensagem",
        )
    )

# COMMAND ----------

# MAGIC %md
# MAGIC ## 1. Execute Bronze Quality Checks

# COMMAND ----------

for entity_name, table_name in BRONZE_ENTITY_TABLES.items():

    run_entity_checks(
        entity_name=entity_name,
        table_name=table_name,
    )

# COMMAND ----------

# MAGIC %md
# MAGIC ## 2. Persist Quality Results

# COMMAND ----------

quality_log_df = build_bronze_quality_log()

quality_log_df.write.mode(
    "append"
).saveAsTable(DATA_QUALITY_LOG_TABLE)

print(
    f"Bronze quality results persisted into: "
    f"{DATA_QUALITY_LOG_TABLE}"
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 3. Display Quality Results

# COMMAND ----------

display(quality_log_df)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 4. Quality Summary

# COMMAND ----------

failed_count = (
    quality_log_df
    .filter("qlt_tx_status_validacao = 'FAILED'")
    .count()
)

warning_count = (
    quality_log_df
    .filter("qlt_tx_status_validacao = 'WARNING'")
    .count()
)

passed_count = (
    quality_log_df
    .filter("qlt_tx_status_validacao = 'PASSED'")
    .count()
)

print("=" * 90)
print("BRONZE QUALITY SUMMARY")
print("=" * 90)
print(f"Passed validations: {passed_count}")
print(f"Warning validations: {warning_count}")
print(f"Failed validations: {failed_count}")
print("=" * 90)

# COMMAND ----------

# ============================================================
# QUALITY EXECUTION POLICY
# ============================================================

if failed_count > 0 and FAIL_ON_ERROR:

    raise Exception(
        f"Bronze quality validation failed with "
        f"{failed_count} failed validation(s)."
    )

if failed_count > 0:

    print(
        f"WARNING: Bronze quality validation finished with "
        f"{failed_count} failed validation(s). "
        "This is expected if Bronze tables have not been created yet."
    )

print("BRONZE QUALITY CHECKS COMPLETED")

BRAZIL LEGISLATIVE ANALYTICS MEDALLION
01 - QUALITY BRONZE CHECKS
Execution Timestamp: 2026-05-20 03:11:00.341811
Catalog: brazil_legislative_analytics
Layer: bronze
Running Bronze quality checks for: brazil_legislative_analytics.bronze.br_deputados
Running Bronze quality checks for: brazil_legislative_analytics.bronze.br_frentes
Running Bronze quality checks for: brazil_legislative_analytics.bronze.br_frentes_membros
Running Bronze quality checks for: brazil_legislative_analytics.bronze.br_frentes_detalhes
Running Bronze quality checks for: brazil_legislative_analytics.bronze.br_eventos
Running Bronze quality checks for: brazil_legislative_analytics.bronze.br_votacoes
Running Bronze quality checks for: brazil_legislative_analytics.bronze.br_votos
Running Bronze quality checks for: brazil_legislative_analytics.bronze.br_despesas_ceap
Running Bronze quality checks for: brazil_legislative_analytics.bronze.br_cpis
Running Bronze quality checks for: brazil_legislative_analytics.bronze.br_c

qlt_id_log,aud_id_execucao,aud_tx_nome_projeto,aud_tx_versao_pipeline,aud_tx_ambiente,aud_tx_nome_notebook,aud_tx_nome_camada,aud_tx_nome_entidade,aud_tx_tabela_destino,qlt_tx_nome_regra,qlt_tx_descricao_regra,qlt_tx_status_validacao,qlt_qt_total_registros,qlt_qt_registros_invalidos,qlt_pc_registros_invalidos,qlt_dh_validacao,qlt_tx_mensagem
94c89b59-79b8-41cb-806f-328383c17a83,b4581acb-ac78-4a69-963e-1f4f5390c58b,brazil_legislative_analytics,v1.0.0,dev,01_quality_bronze_checks,bronze,deputados,brazil_legislative_analytics.bronze.br_deputados,bronze_table_exists,Validates whether the Bronze table exists.,FAILED,1,1,100.0,2026-05-20T03:11:04.431Z,Bronze table does not exist.
db8ec0cb-f1b8-4250-ba29-2b50b16be03e,b4581acb-ac78-4a69-963e-1f4f5390c58b,brazil_legislative_analytics,v1.0.0,dev,01_quality_bronze_checks,bronze,frentes,brazil_legislative_analytics.bronze.br_frentes,bronze_table_exists,Validates whether the Bronze table exists.,FAILED,1,1,100.0,2026-05-20T03:11:04.431Z,Bronze table does not exist.
8450a16c-aa3e-48bc-b1b4-97d7c11e55d1,b4581acb-ac78-4a69-963e-1f4f5390c58b,brazil_legislative_analytics,v1.0.0,dev,01_quality_bronze_checks,bronze,frentes_membros,brazil_legislative_analytics.bronze.br_frentes_membros,bronze_table_exists,Validates whether the Bronze table exists.,FAILED,1,1,100.0,2026-05-20T03:11:04.431Z,Bronze table does not exist.
2986810f-eb9e-474d-a8f8-41c773d2ae3e,b4581acb-ac78-4a69-963e-1f4f5390c58b,brazil_legislative_analytics,v1.0.0,dev,01_quality_bronze_checks,bronze,frentes_detalhes,brazil_legislative_analytics.bronze.br_frentes_detalhes,bronze_table_exists,Validates whether the Bronze table exists.,FAILED,1,1,100.0,2026-05-20T03:11:04.431Z,Bronze table does not exist.
5e107939-fe9e-4a1d-91b4-84e1be23a704,b4581acb-ac78-4a69-963e-1f4f5390c58b,brazil_legislative_analytics,v1.0.0,dev,01_quality_bronze_checks,bronze,eventos,brazil_legislative_analytics.bronze.br_eventos,bronze_table_exists,Validates whether the Bronze table exists.,FAILED,1,1,100.0,2026-05-20T03:11:04.431Z,Bronze table does not exist.
18fa5efa-6c8e-4614-bfc2-dd8611e9735f,b4581acb-ac78-4a69-963e-1f4f5390c58b,brazil_legislative_analytics,v1.0.0,dev,01_quality_bronze_checks,bronze,votacoes,brazil_legislative_analytics.bronze.br_votacoes,bronze_table_exists,Validates whether the Bronze table exists.,FAILED,1,1,100.0,2026-05-20T03:11:04.431Z,Bronze table does not exist.
c9faf5a7-4034-46cb-a948-288be98afc52,b4581acb-ac78-4a69-963e-1f4f5390c58b,brazil_legislative_analytics,v1.0.0,dev,01_quality_bronze_checks,bronze,votos,brazil_legislative_analytics.bronze.br_votos,bronze_table_exists,Validates whether the Bronze table exists.,FAILED,1,1,100.0,2026-05-20T03:11:04.431Z,Bronze table does not exist.
5e26de17-42c1-4906-83d0-12682f537255,b4581acb-ac78-4a69-963e-1f4f5390c58b,brazil_legislative_analytics,v1.0.0,dev,01_quality_bronze_checks,bronze,despesas_ceap,brazil_legislative_analytics.bronze.br_despesas_ceap,bronze_table_exists,Validates whether the Bronze table exists.,FAILED,1,1,100.0,2026-05-20T03:11:04.431Z,Bronze table does not exist.
3c705a3d-17af-497e-876e-242c7523eac8,b4581acb-ac78-4a69-963e-1f4f5390c58b,brazil_legislative_analytics,v1.0.0,dev,01_quality_bronze_checks,bronze,cpis,brazil_legislative_analytics.bronze.br_cpis,bronze_table_exists,Validates whether the Bronze table exists.,FAILED,1,1,100.0,2026-05-20T03:11:04.431Z,Bronze table does not exist.
c4bd4e5a-da79-42a6-8e27-70d111ff5c5c,b4581acb-ac78-4a69-963e-1f4f5390c58b,brazil_legislative_analytics,v1.0.0,dev,01_quality_bronze_checks,bronze,cpi_eventos,brazil_legislative_analytics.bronze.br_cpi_eventos,bronze_table_exists,Validates whether the Bronze table exists.,FAILED,1,1,100.0,2026-05-20T03:11:04.431Z,Bronze table does not exist.


BRONZE QUALITY SUMMARY
Passed validations: 0
Warning validations: 0
Failed validations: 11
BRONZE QUALITY CHECKS COMPLETED
